# EDA: Predict Future Sales

Análisis exploratorio del dataset de la competencia de Kaggle. El objetivo de forecasting es predecir unidades mensuales por par tienda-producto para noviembre de 2015.

## Criterios definidos
- Las devoluciones (`item_cnt_day < 0`) se auditan por separado y no forman parte del análisis principal de ventas.
- Los registros con `item_price <= 0` se reportan y excluyen del análisis principal.
- Los duplicados exactos se reportan y se excluyen de la copia analítica para evitar doble conteo; los CSV de origen permanecen intactos.
- Los outliers se detectan con IQR (1.5 × IQR), se documentan y no se eliminan.
- No se imputan valores faltantes sin evidencia; cada métrica usa observaciones válidas para sus columnas.
- N = 2 para rankings y selección de sucursales, productos y pares en ACF/PACF, según lo acordado.
- Los resultados cuantitativos se generan al ejecutar contra `data/raw/`. No se incluyen cifras hasta disponer de los datos reales.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "sales_train.csv",
    "items.csv",
    "item_categories.csv",
    "shops.csv",
    "test.csv",
]
missing_files = [name for name in REQUIRED_FILES if not (RAW_DIR / name).is_file()]
DATA_AVAILABLE = not missing_files

if DATA_AVAILABLE:
    sales = pd.read_csv(RAW_DIR / "sales_train.csv")
    items = pd.read_csv(RAW_DIR / "items.csv")
    item_categories = pd.read_csv(RAW_DIR / "item_categories.csv")
    shops = pd.read_csv(RAW_DIR / "shops.csv")
    test = pd.read_csv(RAW_DIR / "test.csv")
    print(f"Archivos cargados desde: {RAW_DIR}")
else:
    print("ANÁLISIS NO EJECUTADO: faltan archivos en data/raw:")
    for filename in missing_files:
        print(f"- {filename}")
    print("Descarga la competencia con notebooks/00_kagglehub_download.ipynb y vuelve a ejecutar este notebook.")

## 1. Comprensión inicial del dataset

`date` es la fecha de la transacción; `date_block_num`, el mes consecutivo (0 = enero de 2013); `shop_id` y `item_id`, identificadores de sucursal y producto; `item_price`, precio unitario; `item_cnt_day`, unidades diarias (objetivo del forecasting); `item_name`, nombre del producto; `item_category_id` y `item_category_name`, identificador y nombre de categoría; `shop_name`, nombre de sucursal. `test.csv` contiene los pares tienda-producto a predecir.

La celda siguiente muestra las primeras filas y dimensiones de las cinco tablas, además del rango temporal y bloques observados.

**Interpretación.** Las dimensiones, fechas y bloques se reportarán con los valores calculados de los CSV. Si los archivos no están disponibles, estos resultados quedan explícitamente pendientes; no se estiman.

In [ ]:
if DATA_AVAILABLE:
    tables = {
        "sales_train": sales,
        "items": items,
        "item_categories": item_categories,
        "shops": shops,
        "test": test,
    }
    for table_name, table in tables.items():
        print(f"{table_name}: {table.shape[0]:,} filas × {table.shape[1]} columnas")
        display(table.head())

    parsed_dates = pd.to_datetime(sales["date"], format="%d.%m.%Y", errors="coerce")
    print("Fecha mínima:", parsed_dates.min())
    print("Fecha máxima:", parsed_dates.max())
    print("Bloques mensuales observados:", sales["date_block_num"].nunique())
else:
    print("Interpretación pendiente: sin CSV no se calculan dimensiones ni rango temporal.")

### Interpretación

Las cinco tablas y su rango temporal se identifican a partir de los CSV. La variable objetivo del proyecto es `item_cnt_day`; los resultados numéricos quedan pendientes mientras los archivos no estén disponibles.

## 2. Validación de la estructura de los datos

Se inspeccionarán tipos y dominios; `date` se convertirá con `%d.%m.%Y`. Se comparará el calendario entre la primera y última transacción, y se verificará que `date_block_num` corresponda al año-mes de la fecha. Para forecasting, la agregación mensual es la escala objetivo; la diaria se conserva para explorar ciclos y preparar variables.

**Interpretación.** La frecuencia recomendada se sustentará en los días observados, consistencia de bloques y objetivo mensual; el resultado queda pendiente de ejecutar con los datos.

In [ ]:
if DATA_AVAILABLE:
    print("Tipos de datos originales por tabla:")
    all_dtypes = pd.concat(
        {table_name: table.dtypes for table_name, table in tables.items()},
        names=["tabla", "variable"],
    ).rename("dtype").to_frame()
    id_columns = {"ID", "shop_id", "item_id", "item_category_id"}
    variable_names = all_dtypes.index.get_level_values("variable")
    all_dtypes["clasificacion"] = [
        "fecha" if variable == "date" else
        "categórica (identificador)" if variable in id_columns else
        "numérica" if pd.api.types.is_numeric_dtype(dtype) else
        "categórica"
        for variable, dtype in zip(variable_names, all_dtypes["dtype"])
    ]
    display(all_dtypes)

    sales["date"] = pd.to_datetime(sales["date"], format="%d.%m.%Y", errors="coerce")
    nat_count = int(sales["date"].isna().sum())
    print("Valores NaT después de convertir date:", nat_count)

    valid_dates = sales["date"].dropna()
    observed_days = pd.DatetimeIndex(valid_dates.dt.normalize().unique())
    full_calendar = pd.date_range(observed_days.min(), observed_days.max(), freq="D")
    missing_days = full_calendar.difference(observed_days)
    expected_blocks = (sales["date"].dt.year - 2013) * 12 + sales["date"].dt.month - 1
    block_mismatch = sales["date"].notna() & sales["date_block_num"].ne(expected_blocks)

    print("Días sin transacciones en el calendario:", len(missing_days))
    print("Fechas con date_block_num inconsistente:", int(block_mismatch.sum()))
    print("Bloques observados:", sorted(sales["date_block_num"].dropna().unique()))
    print("Conclusión de diseño: agregación mensual para el objetivo; diaria como diagnóstico y para variables temporales.")
else:
    print("Validación temporal pendiente: se requieren los CSV originales.")

### Interpretación

La agregación mensual es la escala recomendada porque coincide con el objetivo de predicción. Los diagnósticos de días ausentes, fechas inválidas y coherencia de bloques deben revisarse antes de modelar; quedan sin cifras hasta ejecutar los datos.

## 3. Valores faltantes, inválidos y duplicados

Se reportarán nulos y porcentajes en las cinco tablas. No se imputará automáticamente. Se contarán precios no positivos, devoluciones, cantidades cero y duplicados exactos. La copia de análisis eliminará duplicados exactos y precios no positivos; las devoluciones quedarán en una tabla separada y fuera del análisis principal. En `shops.csv` se normalizarán mayúsculas, Unicode y espacios para buscar nombres asociados a varios IDs.

**Interpretación.** Las decisiones de limpieza son conservadoras y reproducibles; los conteos de impacto solo se completarán tras ejecutar el bloque con los CSV.

In [ ]:
if DATA_AVAILABLE:
    for table_name, table in tables.items():
        missing = pd.DataFrame({
            "nulos": table.isna().sum(),
            "porcentaje": table.isna().mean().mul(100).round(4),
        })
        print(f"Valores faltantes: {table_name}")
        display(missing)

    exact_duplicates = int(sales.duplicated().sum())
    invalid_price = int(sales["item_price"].le(0).sum())
    negative_units = int(sales["item_cnt_day"].lt(0).sum())
    zero_units = int(sales["item_cnt_day"].eq(0).sum())
    print("Duplicados exactos:", exact_duplicates)
    print("Registros con item_price <= 0:", invalid_price)
    print("Registros con item_cnt_day < 0 (devoluciones):", negative_units)
    print("Registros con item_cnt_day = 0:", zero_units)

    normalized_shops = shops.assign(
        shop_name_normalized=(
            shops["shop_name"].fillna("").astype(str).str.normalize("NFKC")
            .str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
        )
    )
    repeated_shop_names = (
        normalized_shops.groupby("shop_name_normalized")["shop_id"]
        .agg(ids="nunique", shop_ids=lambda values: sorted(values.unique()))
        .query("ids > 1")
    )
    print("Nombres normalizados asociados a varios shop_id:", len(repeated_shop_names))
    display(repeated_shop_names)

    sales_dedup = sales.drop_duplicates().copy()
    sales_valid_price = sales_dedup.loc[sales_dedup["item_price"].gt(0)].copy()
    sales_valid_price["revenue"] = sales_valid_price["item_price"] * sales_valid_price["item_cnt_day"]
    sales_main = sales_valid_price.loc[sales_valid_price["item_cnt_day"].gt(0)].copy()
    sales_returns = sales_valid_price.loc[sales_valid_price["item_cnt_day"].lt(0)].copy()
    print("Filas para ventas positivas:", f"{len(sales_main):,}")
    print("Filas separadas como devoluciones:", f"{len(sales_returns):,}")
else:
    print("Auditoría pendiente: faltan los CSV para reportar nulos, inválidos y duplicados.")

### Interpretación

La política preserva los CSV originales, evita doble conteo por duplicados exactos y separa devoluciones. Los conteos de nulos, tiendas repetidas y registros inválidos no pueden afirmarse hasta ejecutar sobre el dataset.

## 4. Análisis descriptivo general

Se calcularán media, mediana, mínimo, máximo y desviación estándar de precio, unidades e ingresos (`item_price × item_cnt_day`) sobre ventas positivas válidas. Las distribuciones se resumirán por producto, categoría, sucursal y periodo diario, semanal, mensual y anual.

**Interpretación.** Las escalas y asimetrías se describirán a partir de las tablas ejecutadas; las devoluciones no se mezclarán con las ventas principales.

In [ ]:
if DATA_AVAILABLE:
    metrics = ["item_price", "item_cnt_day", "revenue"]
    print("Estadísticos descriptivos de ventas positivas:")
    display(sales_main[metrics].agg(["mean", "median", "min", "max", "std"]).T)

    sales_enriched = sales_main.merge(
        items[["item_id", "item_category_id"]], on="item_id", how="left"
    ).merge(shops[["shop_id", "shop_name"]], on="shop_id", how="left")

    for label, keys in [
        ("Producto", ["item_id"]),
        ("Categoría", ["item_category_id"]),
        ("Sucursal", ["shop_id", "shop_name"]),
    ]:
        distribution = sales_enriched.groupby(keys, dropna=False).agg(
            unidades=("item_cnt_day", "sum"), ingresos=("revenue", "sum"), transacciones=("item_id", "size")
        ).sort_values("unidades", ascending=False)
        print(f"Distribución por {label} (primeras filas):")
        display(distribution.head())

    for period_name, period in [
        ("Día", sales_main["date"].dt.date),
        ("Semana", sales_main["date"].dt.to_period("W")),
        ("Mes", sales_main["date"].dt.to_period("M")),
        ("Año", sales_main["date"].dt.year),
    ]:
        period_summary = sales_main.assign(period=period).groupby("period").agg(
            unidades=("item_cnt_day", "sum"), ingresos=("revenue", "sum")
        )
        print(f"Distribución temporal por {period_name}: {period_summary.shape[0]} periodos")
        display(period_summary.head())
else:
    print("Descriptivos pendientes: no hay datos para calcular precio, unidades o ingresos.")

### Interpretación

Las métricas principales se calculan sobre transacciones positivas con precio válido, mientras que los ingresos conservan la definición `item_price × item_cnt_day`. Sin datos no es posible comparar dispersión ni concentración por grupo o periodo.

## 5. Exploración temporal

Se generarán series diarias y mensuales de unidades positivas, una tendencia móvil de tres meses, cambios mensuales más pronunciados y promedios por día de semana y mes del año. El análisis separa tendencia, estacionalidad y cambios abruptos sin atribuir causalidad.

**Interpretación.** Las conclusiones sobre dirección, ciclos y picos se redactarán usando las curvas y tablas resultantes. Sin ejecutar con los CSV, no se afirma una tendencia observada.

### Interpretación

La media móvil resume variación de corto plazo, mientras que los promedios por calendario permiten explorar ciclos. No se etiquetan tendencias, estacionalidad ni picos hasta observar los resultados ejecutados.

In [ ]:
if DATA_AVAILABLE:
    daily_series = (
        sales_main.set_index("date")["item_cnt_day"].resample("D").sum().asfreq("D", fill_value=0)
    )
    monthly_series = (
        sales_main.groupby("date_block_num")["item_cnt_day"].sum().reindex(range(34), fill_value=0)
    )
    monthly_series.index.name = "date_block_num"

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)
    daily_series.plot(ax=axes[0], color="#187b7a", linewidth=0.8)
    axes[0].set(title="Ventas diarias (unidades positivas)", xlabel="Fecha", ylabel="Unidades")
    monthly_series.plot(ax=axes[1], marker="o", color="#d16c4b")
    monthly_series.rolling(3, min_periods=1).mean().plot(
        ax=axes[1], color="#354b73", linestyle="--", label="Media móvil (3 meses)"
    )
    axes[1].set(title="Ventas mensuales", xlabel="Bloque mensual", ylabel="Unidades")
    axes[1].legend()
    fig.savefig(FIGURES_DIR / "ventas_totales_mensuales_y_diarias.png", dpi=160, bbox_inches="tight")
    plt.show()

    monthly_changes = monthly_series.diff().dropna().sort_values()
    print("Mayores caídas mensuales (diferencia absoluta):")
    display(monthly_changes.head(3))
    print("Mayores aumentos mensuales (diferencia absoluta):")
    display(monthly_changes.tail(3).sort_values(ascending=False))

    weekday_names = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
    weekday_mean = sales_main.groupby(sales_main["date"].dt.dayofweek)["item_cnt_day"].mean()
    weekday_mean.index = [weekday_names[index] for index in weekday_mean.index]
    month_mean = sales_main.groupby(sales_main["date"].dt.month)["item_cnt_day"].mean()
    print("Promedio de unidades por día de semana:")
    display(weekday_mean.to_frame("media_unidades"))
    print("Promedio de unidades por mes del año:")
    display(month_mean.to_frame("media_unidades"))
else:
    print("Exploración temporal pendiente: se requiere sales_train.csv para crear series y gráficas.")

## 6. Comparativo entre sucursales y productos

Se calcularán rankings top y bottom N = 2 para productos y categorías por unidades e ingresos, además del ranking de sucursales. Los outliers transaccionales y de agregados por sucursal/producto se marcarán con la regla de Tukey: valores fuera de `[Q1 − 1.5 × IQR, Q3 + 1.5 × IQR]`. Se conservarán para no distorsionar el análisis.

**Interpretación.** Las tablas permitirán distinguir concentración de ventas y observaciones atípicas; ningún producto, tienda o transacción se excluye por ser outlier.

In [ ]:
if DATA_AVAILABLE:
    TOP_N = 2
    product_summary = sales_enriched.groupby("item_id").agg(
        unidades=("item_cnt_day", "sum"), ingresos=("revenue", "sum")
    ).reset_index().merge(items[["item_id", "item_name"]], on="item_id", how="left")
    category_summary = sales_enriched.groupby("item_category_id").agg(
        unidades=("item_cnt_day", "sum"), ingresos=("revenue", "sum")
    ).reset_index().merge(
        item_categories[["item_category_id", "item_category_name"]],
        on="item_category_id", how="left",
    )
    shop_summary = sales_enriched.groupby(["shop_id", "shop_name"]).agg(
        unidades=("item_cnt_day", "sum"), ingresos=("revenue", "sum")
    ).sort_values("unidades", ascending=False)

    for label, summary, key in [
        ("Productos", product_summary, "item_name"),
        ("Categorías", category_summary, "item_category_name"),
    ]:
        print(f"{label}: top y bottom {TOP_N} por unidades")
        display(summary.nlargest(TOP_N, "unidades")[[key, "unidades", "ingresos"]])
        display(summary.nsmallest(TOP_N, "unidades")[[key, "unidades", "ingresos"]])
        print(f"{label}: top y bottom {TOP_N} por ingresos")
        display(summary.nlargest(TOP_N, "ingresos")[[key, "unidades", "ingresos"]])
        display(summary.nsmallest(TOP_N, "ingresos")[[key, "unidades", "ingresos"]])

    print("Ranking de sucursales por unidades e ingresos:")
    display(shop_summary)

    for column in ["item_cnt_day", "item_price"]:
        values = sales_main[column].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        flagged = sales_main[column].lt(lower) | sales_main[column].gt(upper)
        print(f"Outliers transaccionales en {column}: {int(flagged.sum()):,}; umbrales=({lower:.4f}, {upper:.4f})")

    for label, summary in [("sucursal", shop_summary.reset_index()), ("producto", product_summary)]:
        q1, q3 = summary["unidades"].quantile([0.25, 0.75])
        iqr = q3 - q1
        flagged = summary.loc[
            summary["unidades"].lt(q1 - 1.5 * iqr) | summary["unidades"].gt(q3 + 1.5 * iqr)
        ].sort_values("unidades", ascending=False)
        print(f"Agregados marcados como outliers por {label}: {len(flagged)}")
        display(flagged.head(10))
else:
    print("Comparativos y outliers pendientes: requieren los CSV de ventas y catálogos.")

### Interpretación

Los rankings muestran por separado volumen e ingresos; el umbral IQR marca casos extremos sin quitarlos. Los productos o tiendas señalados no se califican como error sin una revisión de negocio.

## 7. Visualizaciones obligatorias

Se guardarán en `reports/figures/`: la serie temporal total, barras de sucursales, top N productos, ventas por categoría y dos matrices mensuales de calor (tiempo × sucursal y tiempo × categoría). Los nombres de archivo y títulos estarán en español.

**Interpretación.** Las figuras facilitarán comparar escala, concentración y cambios temporales. Las pautas concretas se describirán al revisar las figuras con datos reales.

In [ ]:
if DATA_AVAILABLE:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    shop_plot = shop_summary.reset_index().sort_values("unidades", ascending=False)
    fig, ax = plt.subplots(figsize=(13, 6))
    sns.barplot(data=shop_plot, x="shop_id", y="unidades", color="#187b7a", ax=ax)
    ax.set(title="Unidades vendidas por sucursal", xlabel="ID de sucursal", ylabel="Unidades")
    ax.tick_params(axis="x", rotation=90)
    fig.savefig(FIGURES_DIR / "ventas_por_sucursal.png", dpi=160, bbox_inches="tight")
    plt.show()

    top_products = product_summary.nlargest(TOP_N, "unidades").sort_values("unidades")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=top_products, x="unidades", y="item_name", color="#d16c4b", ax=ax)
    ax.set(title=f"Top {TOP_N} productos por unidades", xlabel="Unidades", ylabel="Producto")
    fig.savefig(FIGURES_DIR / "top_productos_por_unidades.png", dpi=160, bbox_inches="tight")
    plt.show()

    top_categories = category_summary.nlargest(TOP_N, "unidades").sort_values("unidades")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=top_categories, x="unidades", y="item_category_name", color="#354b73", ax=ax)
    ax.set(title=f"Categorías por unidades (Top {TOP_N})", xlabel="Unidades", ylabel="Categoría")
    fig.savefig(FIGURES_DIR / "ventas_por_categoria.png", dpi=160, bbox_inches="tight")
    plt.show()

    monthly_shop = sales_enriched.groupby(["date_block_num", "shop_id"])["item_cnt_day"].sum().unstack(fill_value=0)
    monthly_shop = monthly_shop.reindex(range(34), fill_value=0)
    monthly_category = sales_enriched.groupby(["date_block_num", "item_category_id"])["item_cnt_day"].sum().unstack(fill_value=0)
    monthly_category = monthly_category.reindex(range(34), fill_value=0)

    fig, ax = plt.subplots(figsize=(18, 8))
    sns.heatmap(monthly_shop.T, cmap="YlGnBu", ax=ax, cbar_kws={"label": "Unidades"})
    ax.set(title="Unidades mensuales por sucursal", xlabel="Bloque mensual", ylabel="ID de sucursal")
    fig.savefig(FIGURES_DIR / "heatmap_tiempo_por_sucursal.png", dpi=160, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(18, 10))
    sns.heatmap(monthly_category.T, cmap="rocket_r", ax=ax, cbar_kws={"label": "Unidades"})
    ax.set(title="Unidades mensuales por categoría", xlabel="Bloque mensual", ylabel="ID de categoría")
    fig.savefig(FIGURES_DIR / "heatmap_tiempo_por_categoria.png", dpi=160, bbox_inches="tight")
    plt.show()
    print("Figuras guardadas en:", FIGURES_DIR)
else:
    print("Visualizaciones no generadas: se requieren los CSV reales.")

### Interpretación

Las figuras contrastan tiendas y segmentos en la misma unidad de medida, y los heatmaps exponen cambios por bloque mensual. No es posible enumerar patrones visuales sin generar las gráficas con los datos.

## 8. Autocorrelación (ACF) y autocorrelación parcial (PACF)

Se analizarán ocho series representativas: ventas totales mensuales y diarias, las dos sucursales con más unidades, los dos productos principales y los dos pares tienda-producto más vendidos. Las series por grupo se agregan en los 34 bloques mensuales; ACF/PACF incluyen bandas de confianza del 95 %.

**Interpretación.** Se consideran indicios los rezagos que exceden sus bandas de confianza; un patrón repetido en rezagos mensuales puede sugerir estacionalidad, pero no demuestra causalidad. La lectura concreta depende de las gráficas calculadas.

In [ ]:
if DATA_AVAILABLE:
    monthly_items = sales_enriched.groupby(["date_block_num", "item_id"])["item_cnt_day"].sum().unstack(fill_value=0)
    monthly_items = monthly_items.reindex(range(34), fill_value=0)
    monthly_pairs = sales_main.groupby(["date_block_num", "shop_id", "item_id"])["item_cnt_day"].sum().unstack(["shop_id", "item_id"], fill_value=0)
    monthly_pairs = monthly_pairs.reindex(range(34), fill_value=0)

    top_shop_ids = shop_summary.reset_index().nlargest(TOP_N, "unidades")["shop_id"].tolist()
    top_item_ids = product_summary.nlargest(TOP_N, "unidades")["item_id"].tolist()
    top_pair_ids = (
        sales_main.groupby(["shop_id", "item_id"])["item_cnt_day"].sum()
        .nlargest(TOP_N).index.tolist()
    )

    series_to_analyze = {
        "total_mensual": monthly_series,
        "total_diario": daily_series,
    }
    for shop_id in top_shop_ids:
        series_to_analyze[f"sucursal_{shop_id}_mensual"] = monthly_shop[shop_id]
    for item_id in top_item_ids:
        series_to_analyze[f"producto_{item_id}_mensual"] = monthly_items[item_id]
    for shop_id, item_id in top_pair_ids:
        series_to_analyze[f"par_sucursal_{shop_id}_producto_{item_id}_mensual"] = monthly_pairs[(shop_id, item_id)]

    print("Series ACF/PACF analizadas:", len(series_to_analyze))
    for series_name, values in series_to_analyze.items():
        values = pd.Series(values, dtype="float64").dropna()
        max_lags = min(40 if "diario" in series_name else 12, len(values) // 2 - 1)
        if max_lags < 1 or values.nunique() < 2:
            print(f"Se omite {series_name}: longitud o variabilidad insuficiente.")
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
        plot_acf(values, lags=max_lags, alpha=0.05, ax=axes[0], zero=False)
        plot_pacf(values, lags=max_lags, alpha=0.05, method="ywm", ax=axes[1], zero=False)
        axes[0].set(title=f"ACF: {series_name}", xlabel="Rezago", ylabel="Correlación")
        axes[1].set(title=f"PACF: {series_name}", xlabel="Rezago", ylabel="Correlación parcial")
        safe_name = series_name.replace(" ", "_")
        fig.savefig(FIGURES_DIR / f"acf_pacf_{safe_name}.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("ACF/PACF pendiente: no se pueden construir las series sin los CSV de Kaggle.")

### Interpretación

Las barras de confianza permiten diferenciar rezagos con evidencia estadística bajo la aproximación del gráfico. En series mensuales cortas (34 observaciones), la interpretación debe ser prudente y validarse fuera de muestra; las gráficas aún no están disponibles.

## 9. Insights para forecasting

Se cuantificará qué proporción de pares únicos `(shop_id, item_id)` de `test.csv` aparece en `sales_train.csv`. También se revisarán señales asociadas con el volumen observado (tienda, producto/categoría, periodo, precio) sin afirmar efectos causales.

**Interpretación.** La cobertura del test se reportará como porcentaje calculado por inner join de pares únicos. Los pares sin historial requieren una estrategia de cold start en la etapa de modelado.

In [ ]:
if DATA_AVAILABLE:
    train_pairs = sales[["shop_id", "item_id"]].drop_duplicates()
    test_pairs = test[["shop_id", "item_id"]].drop_duplicates()
    history_check = test_pairs.merge(
        train_pairs.assign(has_history=True), on=["shop_id", "item_id"], how="left"
    )
    history_check["has_history"] = history_check["has_history"].fillna(False)
    history_count = int(history_check["has_history"].sum())
    test_pair_count = len(test_pairs)
    history_percent = 100 * history_count / test_pair_count if test_pair_count else np.nan
    print(f"Pares únicos de test con historial: {history_count:,} / {test_pair_count:,} ({history_percent:.2f}%)")
    print("Pares sin historial:", int((~history_check["has_history"]).sum()))

    evidence = pd.DataFrame({
        "variable": ["sucursal", "producto", "categoría", "periodo", "precio"],
        "evidencia_disponible": [
            "ranking y variación entre sucursales",
            "ranking y concentración por item_id",
            "agregación por item_category_id",
            "series por fecha y date_block_num",
            "resúmenes de precio en transacciones válidas",
        ],
        "alcance": "asociación descriptiva, no causal",
    })
    display(evidence)
else:
    print("Insights y cobertura de historial pendientes: se necesitan train y test reales.")

### Interpretación

El porcentaje de cobertura se basa en pares únicos y un cruce interno por tienda-producto, por lo que responde directamente qué fracción de test tiene alguna fila en train. La cifra requiere ambos CSV.

## 10. Conclusiones y preparación del dataset

La conclusión evaluará faltantes, consistencia temporal, registros inválidos, devoluciones, duplicados, outliers y cobertura de historial en test. Para el forecasting mensual se recomienda construir una cuadrícula completa tienda-producto-mes, asignar cero solo a combinaciones sin transacción después de validar la semántica y separar explícitamente pares nuevos.

**Interpretación.** La calidad final y la preparación se dictaminarán con los diagnósticos ejecutados. Hasta descargar y validar los CSV, no se declara que el dataset esté limpio ni listo para modelar.

In [ ]:
if DATA_AVAILABLE:
    quality_summary = pd.DataFrame({
        "diagnóstico": [
            "Filas originales de train",
            "Nulos en train",
            "Duplicados exactos en train",
            "Precios no positivos en train",
            "Devoluciones en train",
            "Fechas inválidas tras conversión",
            "Desajustes de date_block_num",
            "Pares únicos de test con historial (%)",
        ],
        "resultado": [
            len(sales),
            int(sales.isna().sum().sum()),
            int(sales.duplicated().sum()),
            int(sales["item_price"].le(0).sum()),
            int(sales["item_cnt_day"].lt(0).sum()),
            int(sales["date"].isna().sum()),
            int(block_mismatch.sum()),
            round(history_percent, 2),
        ],
    })
    display(quality_summary)
    print("Preparación propuesta: usar ventas válidas y positivas; conservar devoluciones y datos crudos aparte;")
    print("documentar outliers sin eliminarlos; validar la cuadrícula mensual y el tratamiento de pares nuevos antes del modelo.")
else:
    print("Conclusión no calculable: KaggleHub no pudo autenticarse y no hay CSV en data/raw.")
    print("La autenticación debe configurarse localmente antes de afirmar patrones, calidad o cobertura del test.")

### Interpretación

La preparación para modelar requiere cerrar los diagnósticos de calidad, generar la matriz tienda-producto-mes y evaluar pares sin historial por separado. Sin la descarga autenticada, el estado del dataset y las conclusiones cuantitativas permanecen sin determinar.